In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

np.random.seed(42)

N_TRANSACTIONS = 120_000

In [2]:
N_TRANSACTIONS

120000

### Load Our Dimension Tables

In [3]:
data_path = Path("../data/raw")

df_store = pd.read_csv(
    data_path / "dim_store.csv"
)

df_product = pd.read_csv(
    data_path / "dim_product.csv"
)

df_customer = pd.read_csv(
    data_path / "dim_customer.csv"
)

df_date = pd.read_csv(
    data_path / "dim_date.csv"
)

### Validate Before Generating Sales

In [5]:
assert df_store["store_id"].is_unique
assert df_product["product_id"].is_unique
assert df_customer["customer_id"].is_unique
assert df_date["date"].is_unique

### Generate Transaction Skeleton

In [6]:
N_TRANSACTIONS = 120_000

transaction_ids = np.arange(1, N_TRANSACTIONS + 1)

df_sales = pd.DataFrame({
    "transaction_id": [
        f"T{i:06d}" for i in transaction_ids
    ]
})

df_sales.head()

,transaction_id
0,T000001
1,T000002
2,T000003
3,T000004
4,T000005


In [7]:
df_sales.shape

(120000, 1)

In [8]:
df_sales["transaction_id"].is_unique

True

### 5. Generate Transaction Dates

In [9]:
df_date["date"] = pd.to_datetime(df_date["date"])

In [10]:
df_date[["date"]].head()

,date
0,2024-01-01
1,2024-01-02
2,2024-01-03
3,2024-01-04
4,2024-01-05


In [11]:
df_date["date"].min(), df_date["date"].max()

(Timestamp('2024-01-01 00:00:00'), Timestamp('2025-12-31 00:00:00'))

### Create Day-of-Week Behavior

In [12]:
date_weights = df_date[["date"]].copy()

date_weights["day_of_week"] = date_weights["date"].dt.dayofweek

### Base Weekday Weight

In [13]:
weekday_weights = {
    0: 1.00,  # Monday
    1: 1.00,  # Tuesday
    2: 1.00,  # Wednesday
    3: 1.05,  # Thursday
    4: 1.20,  # Friday
    5: 1.30,  # Saturday
    6: 1.10   # Sunday
}

In [14]:
date_weights["weekday_weight"] = (
    date_weights["day_of_week"]
    .map(weekday_weights)
)

In [15]:
date_weights.head(10)

,date,day_of_week,weekday_weight
0,2024-01-01,0,1.00
1,2024-01-02,1,1.00
2,2024-01-03,2,1.00
3,2024-01-04,3,1.05
4,2024-01-05,4,1.20
5,2024-01-06,5,1.30
6,2024-01-07,6,1.10
7,2024-01-08,0,1.00
8,2024-01-09,1,1.00
9,2024-01-10,2,1.00


### Add Monthly Seasonality

In [16]:
date_weights["month"] = date_weights["date"].dt.month

In [17]:
monthly_weights = {
    1: 0.95,
    2: 0.95,
    3: 1.05,
    4: 1.00,
    5: 1.00,
    6: 1.05,
    7: 1.00,
    8: 1.00,
    9: 1.05,
    10: 1.10,
    11: 1.15,
    12: 1.20
}

In [18]:
date_weights["seasonal_weight"] = (
    date_weights["month"].map(monthly_weights)
)

### Add 2025 Growth

In [19]:
date_weights["year"] = date_weights["date"].dt.year

date_weights["year_weight"] = np.where(
    date_weights["year"] == 2025,
    1.08,
    1.00
)

### Combine the Weights

In [20]:
date_weights["weight"] = (
    date_weights["weekday_weight"]
    * date_weights["seasonal_weight"]
    * date_weights["year_weight"]
)

In [21]:
date_weights[
    [
        "date",
        "weekday_weight",
        "seasonal_weight",
        "year_weight",
        "weight"
    ]
].head(10)

,date,weekday_weight,seasonal_weight,year_weight,weight
0,2024-01-01,1.00,0.95,1.0,0.9500
1,2024-01-02,1.00,0.95,1.0,0.9500
2,2024-01-03,1.00,0.95,1.0,0.9500
3,2024-01-04,1.05,0.95,1.0,0.9975
4,2024-01-05,1.20,0.95,1.0,1.1400
5,2024-01-06,1.30,0.95,1.0,1.2350
6,2024-01-07,1.10,0.95,1.0,1.0450
7,2024-01-08,1.00,0.95,1.0,0.9500
8,2024-01-09,1.00,0.95,1.0,0.9500
9,2024-01-10,1.00,0.95,1.0,0.9500


### Convert Weights to Probabilities

In [22]:
date_weights["probability"] = (
    date_weights["weight"]
    / date_weights["weight"].sum()
)

In [23]:
date_weights["probability"].sum()

np.float64(1.0)

In [24]:
np.isclose(
    date_weights["probability"].sum(),
    1.0
)

np.True_

### Generate 120,000 Dates

In [25]:
df_sales["transaction_date"] = np.random.choice(
    date_weights["date"],
    size=N_TRANSACTIONS,
    p=date_weights["probability"]
)

In [26]:
df_sales.head()

,transaction_id,transaction_date
0,T000001,2024-10-20
1,T000002,2025-12-01
2,T000003,2025-07-03
3,T000004,2025-03-29
4,T000005,2024-05-04


### Validate the Dates

In [27]:
df_sales["transaction_date"].min()
df_sales["transaction_date"].max()

Timestamp('2025-12-31 00:00:00')

In [28]:
df_sales["transaction_date"].isna().sum()

np.int64(0)

In [29]:
df_sales.shape

(120000, 2)

### Inspect Daily Distribution

In [30]:
daily_sales_lines = (
    df_sales
    .groupby("transaction_date")
    .size()
    .reset_index(name="transaction_lines")
)

In [31]:
daily_sales_lines.head()

,transaction_date,transaction_lines
0,2024-01-01,133
1,2024-01-02,129
2,2024-01-03,131
3,2024-01-04,153
4,2024-01-05,141


In [32]:
daily_sales_lines["transaction_lines"].describe()

count    731.000000
mean     164.158687
std       23.903619
min      105.000000
25%      146.000000
50%      163.000000
75%      179.000000
max      256.000000
Name: transaction_lines, dtype: float64

### Check Monthly Distribution

In [33]:
monthly_distribution = (
    df_sales
    .assign(
        year=df_sales["transaction_date"].dt.year,
        month=df_sales["transaction_date"].dt.month
    )
    .groupby(["year", "month"])
    .size()
    .reset_index(name="transaction_lines")
)

In [34]:
monthly_distribution

,year,month,transaction_lines
0,2024,1,4365
1,2024,2,4276
2,2024,3,5012
3,2024,4,4494
4,2024,5,4702
5,2024,6,4816
6,2024,7,4642
7,2024,8,4640
8,2024,9,4942
9,2024,10,4924


### Compare 2024 vs 2025

In [35]:
year_distribution = (
    df_sales["transaction_date"]
    .dt.year
    .value_counts()
    .sort_index()
)

year_distribution

transaction_date
2024    57675
2025    62325
Name: count, dtype: int64

### Check Day-of-Week Distribution

In [36]:
dow_distribution = (
    df_sales["transaction_date"]
    .dt.day_name()
    .value_counts()
)

In [37]:
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

dow_distribution = dow_distribution.reindex(day_order)

dow_distribution

transaction_date
Monday       15808
Tuesday      15863
Wednesday    15938
Thursday     16262
Friday       18679
Saturday     20256
Sunday       17194
Name: count, dtype: int64